In [1]:
from datetime import datetime

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import (
    RFECV,
)
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split, KFold
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression

In [2]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

EXTRACT_TRAIN_PARQUET = f"{OUTPUT_PREFIX}/extract_train_v2.parquet"
EXTRACT_TEST_PARQUET = f"{OUTPUT_PREFIX}/extract_test_v2.parquet"
SUBMISSION_CSV = f"{DATA_PREFIX}/sample_submission.csv"


In [3]:
BANDS_S2 = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
# BANDS_S2_DIST = [f"{b}_DIST" for b in BANDS_S2]
BANDS_S1 = ["VV", "VH"]
BANDS_CHM = [
    # "CHM",
    "CHM_META",
    "CHM_ETH",
    "CHM_ALS",
]
BANDS_TERRAIN = [
    # "elevation",
    "slope",
    "aspect",
    "tpi",
    "tri",
    "hillshade",
]
BANDS_LST = ["LST"]
BANDS_TREE = ["CANOPY_DENSITY"]
# BANDS_S1_DIST = [f"{b}_DIST" for b in BANDS_S1]

INDICES = [
    dict(name="NDVI", band1="NIR", band2="RED"),
    dict(name="NDMI", band1="NIR", band2="SWIR1"),
    dict(name="NBR", band1="NIR", band2="SWIR2"),
    dict(name="NBR2", band1="SWIR1", band2="SWIR2"),
    dict(name="NDWI", band1="GREEN", band2="NIR"),
    dict(name="MNDWI", band1="GREEN", band2="SWIR1"),
    dict(name="MNDWI2", band1="GREEN", band2="SWIR2"),
    dict(name="RVI", band1="VV", band2="VH"),
]

INDICES_BANDS = [indi["name"] for indi in INDICES]
# INDICES_BANDS_DIST = [f"{b}_DIST" for b in INDICES_BANDS]

PREDICTORS = [
    *BANDS_S2,
    *BANDS_S1,
    *INDICES_BANDS,
    *BANDS_CHM,
    *BANDS_TREE,
    *BANDS_TERRAIN,
    *BANDS_LST,
    # *BANDS_S2_DIST,
    # *BANDS_S1_DIST,
    # *INDICES_BANDS_DIST,
]

LABEL = "biomass"

PREDICTORS

['BLUE',
 'GREEN',
 'RED',
 'NIR',
 'SWIR1',
 'SWIR2',
 'VV',
 'VH',
 'NDVI',
 'NDMI',
 'NBR',
 'NBR2',
 'NDWI',
 'MNDWI',
 'MNDWI2',
 'RVI',
 'CHM_META',
 'CHM_ETH',
 'CHM_ALS',
 'CANOPY_DENSITY',
 'slope',
 'aspect',
 'tpi',
 'tri',
 'hillshade',
 'LST']

In [4]:
# load parquet extracted
train_df = gpd.read_parquet(EXTRACT_TRAIN_PARQUET).dropna()
train_df

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,CANOPY_DENSITY,CHM_ETH,CHM_META,CHM_ALS,slope,aspect,tri,tpi,hillshade,LST
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,51,11,5,0.000000,19.786716,67.750977,14.730920,0.125,150.0,42.659630
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,74,15,8,0.000000,24.269398,337.833649,21.166010,-3.750,209.0,35.854351
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,46,13,5,0.000000,22.094799,356.760315,16.062378,-0.250,198.0,40.349049
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,40,13,0,0.000000,22.413956,211.328705,16.401220,-0.375,185.0,43.090302
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,70,13,7,0.000000,11.787092,222.137589,9.219544,0.375,189.0,38.400776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148498,3111814.7416963745,2243943.9191588927,2019,96.743500,035805,POINT (-4.70358 42.23347),371.0,607.0,566.0,2937.0,...,62,13,4,13.893074,3.580432,324.462341,2.828427,0.250,186.0,26.765837
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,56,16,0,14.550603,2.943096,315.000000,3.464102,0.750,185.0,26.075397
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,82,12,2,13.315552,1.316193,71.565048,1.414214,0.000,179.0,24.615902
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,79,12,4,16.413029,2.426940,120.963760,2.236068,0.125,177.0,24.615902


In [5]:
# Generating indices
def generate_indices(table, band_suffix=""):
    for index_dict in INDICES:
        name = f"{index_dict['name']}{band_suffix}"
        band1 = f"{index_dict['band1']}{band_suffix}"
        band2 = f"{index_dict['band2']}{band_suffix}"
        table[name] = (
            ((table[band1] / 1e4) - (table[band2] / 1e4))
            / ((table[band1] / 1e4) + (table[band2] / 1e4))
            * 1e4
        )


generate_indices(train_df)
# generate_indices(train_df, "_DIST")

train_df

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,hillshade,LST,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,150.0,42.659630,6082.148499,953.303206,3286.348212,2408.500590,-6246.409192,-5628.252788,-3724.653148,5634.005764
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,209.0,35.854351,7891.138575,2368.660244,5231.699127,3268.015171,-7260.273973,-5907.540735,-3271.028037,5745.654163
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,198.0,40.349049,5931.465869,908.752328,3172.925568,2331.396817,-6097.829074,-5493.494129,-3626.570916,5230.875258
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,185.0,43.090302,3526.570048,-428.312890,1646.684610,2060.465116,-4745.972739,-5071.200232,-3362.035225,4729.829756
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,189.0,38.400776,7243.589744,2396.313364,4677.174901,2568.768515,-6497.444634,-4857.428714,-2614.942529,4161.585366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148498,3111814.7416963745,2243943.9191588927,2019,96.743500,035805,POINT (-4.70358 42.23347),371.0,607.0,566.0,2937.0,...,186.0,26.765837,6768.484156,1165.177723,3695.500117,2644.178455,-6574.492099,-5858.068918,-3802.960694,4750.430293
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,185.0,26.075397,4213.197970,1301.259283,3425.393172,2223.230490,-4730.639731,-3654.333502,-1557.653405,6197.623515
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,179.0,24.615902,6973.029589,1990.381058,4546.678636,2810.650888,-7035.479632,-5866.617809,-3659.359191,4710.860367
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,177.0,24.615902,7311.254561,2233.240777,4431.445952,2439.644219,-7161.936561,-5867.098865,-4000.000000,5334.750266


In [6]:
train_df_filter = train_df[(train_df["BLUE"] > 0) & (train_df["VV"] > 0)]
train_df_filter

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,hillshade,LST,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217),366.0,588.0,620.0,2545.0,...,150.0,42.659630,6082.148499,953.303206,3286.348212,2408.500590,-6246.409192,-5628.252788,-3724.653148,5634.005764
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357),305.0,540.0,401.0,3402.0,...,209.0,35.854351,7891.138575,2368.660244,5231.699127,3268.015171,-7260.273973,-5907.540735,-3271.028037,5745.654163
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368),464.0,710.0,748.0,2929.0,...,198.0,40.349049,5931.465869,908.752328,3172.925568,2331.396817,-6097.829074,-5493.494129,-3626.570916,5230.875258
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331),622.0,848.0,1139.0,2380.0,...,185.0,43.090302,3526.570048,-428.312890,1646.684610,2060.465116,-4745.972739,-5071.200232,-3362.035225,4729.829756
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668),281.0,514.0,387.0,2421.0,...,189.0,38.400776,7243.589744,2396.313364,4677.174901,2568.768515,-6497.444634,-4857.428714,-2614.942529,4161.585366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5148498,3111814.7416963745,2243943.9191588927,2019,96.743500,035805,POINT (-4.70358 42.23347),371.0,607.0,566.0,2937.0,...,186.0,26.765837,6768.484156,1165.177723,3695.500117,2644.178455,-6574.492099,-5858.068918,-3802.960694,4750.430293
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312),882.0,1252.0,1425.0,3500.0,...,185.0,26.075397,4213.197970,1301.259283,3425.393172,2223.230490,-4730.639731,-3654.333502,-1557.653405,6197.623515
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345),380.0,564.0,578.0,3241.0,...,179.0,24.615902,6973.029589,1990.381058,4546.678636,2810.650888,-7035.479632,-5866.617809,-3659.359191,4710.860367
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471),307.0,510.0,479.0,3084.0,...,177.0,24.615902,7311.254561,2233.240777,4431.445952,2439.644219,-7161.936561,-5867.098865,-4000.000000,5334.750266


In [ ]:
# generate correlation matrix
correlation = train_df[[LABEL, *PREDICTORS]].corr()

plt.figure(figsize=(15, 15))
plt.imshow(correlation)
plt.title("Correlation Matrix")
plt.xticks(range(len([LABEL, *PREDICTORS])), [LABEL, *PREDICTORS], size=8, rotation=90)
plt.yticks(range(len([LABEL, *PREDICTORS])), [LABEL, *PREDICTORS], size=8)
plt.show()


In [ ]:
# split train and test data
train, test = train_test_split(train_df_filter, test_size=0.3)

In [ ]:
# train limit sample
train_limit = train.sample(10_000)
test_limit = test.sample(1000)
train_limit

,x,y,year,biomass,tile_id,geometry,BLUE,GREEN,RED,NIR,...,hillshade,LST,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
4942638,3116444.7416963745,2258003.9191588927,2019,134.430420,036108,POINT (-4.67917 42.36586),458.0,762.0,795.0,3773.0,...,176.0,29.578869,6519.264448,3727.487721,5929.913447,2827.380952,-6639.470783,-3869.670153,-1170.336037,5713.570356
2855014,3691844.7416963745,2103813.9191588927,2016,104.566795,093465,POINT (2.44978 41.75362),216.0,441.0,232.0,3506.0,...,213.0,27.519510,8758.694489,3126.169974,6364.060677,4042.065010,-7765.391437,-6126.482213,-2770.491803,5428.571429
1719173,3661944.7416963745,2129663.9191588927,2016,116.648163,090484,POINT (2.0629 41.95815),233.0,513.0,303.0,3267.0,...,162.0,36.907101,8302.521008,3383.859074,6038.291605,3336.085879,-7285.714286,-5178.571429,-2227.272727,6070.941337
3556909,3540044.7416963745,2024353.9191588927,2016,21.518366,078190,POINT (0.74912 40.88795),829.0,1247.0,1561.0,3328.0,...,173.0,36.667839,3614.236040,-388.447653,1327.433628,1707.078926,-4548.633880,-4851.362510,-3428.194993,4705.882353
1928571,3514994.7416963745,2059053.9191588927,2016,4.603819,075809,POINT (0.4071 41.17089),1082.0,1507.0,1790.0,3012.0,...,191.0,38.790432,2544.773011,-759.318914,641.229465,1393.762183,-3330.382828,-3988.831272,-2747.834456,5327.679260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4127487,3160424.7416963745,2306473.9191588927,2019,117.189842,040610,POINT (-4.25749 42.87044),317.0,611.0,369.0,4129.0,...,199.0,22.404444,8359.270787,3627.062706,6552.415314,3837.334289,-7421.940928,-5192.761605,-1692.726037,6214.715410
947858,3498464.7416963745,2016703.9191588927,2016,63.309193,074300,POINT (0.27156 40.77234),468.0,829.0,766.0,2624.0,...,223.0,26.835907,5480.825959,1295.738269,3391.171217,2191.739524,-5198.378222,-4184.496668,-2193.973635,5291.987325
4893319,3174174.7416963745,2248323.9191588927,2019,57.600899,042085,POINT (-3.96917 42.37897),286.0,446.0,356.0,2505.0,...,179.0,29.913834,7511.359664,1517.241379,4446.366782,3141.025641,-6977.295832,-6106.503710,-3669.268985,5163.002274
1310126,3687214.7416963745,2142213.9191588927,2016,115.221146,093179,POINT (2.35126 42.09441),180.0,442.0,239.0,3536.0,...,128.0,30.190693,8733.774834,3581.716920,6774.193548,4215.227563,-7777.777778,-5816.374823,-2121.212121,5557.195572


In [ ]:
min_features_to_select = 3
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rfecv = RFECV(
    estimator=LinearRegression(),
    step=1,
    cv=cv,
    min_features_to_select=min_features_to_select,
    scoring="neg_mean_squared_error",
)
rfecv.fit(train_limit[PREDICTORS], train_limit[LABEL])

,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,LinearRegression()
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",3
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.",'neg_mean_squared_error'
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
"cv_results_ cv_results_: dict of ndarraysAll arrays (values of the dictionary) are sorted in ascending orderby the number of features used (i.e., the first element of the arrayrepresents the models that used the least number of features, while thelast element represents the models that used all available features)... versionadded:: 1.0This dictionary contains the following keys:split(k)_test_score : ndarray of shape (n_subsets_of_features,) The cross-validation scores across (k)th fold.mean_test_score : ndarray of shape (n_subsets_of_features,) Mean of scores over the folds.std_test_score : ndarray of shape (n_subsets_of_features,) Standard deviation of scores over the folds.n_features : ndarray of shape (n_subsets_of_features,) Number of features used at each step

In [ ]:
SELECTED_PREDICTORS = np.array(PREDICTORS)[rfecv.support_]
SELECTED_PREDICTORS

array(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'VV', 'VH',
       'NDVI', 'NDMI', 'NBR', 'NBR2', 'NDWI', 'MNDWI', 'MNDWI2', 'RVI',
       'CHM_META', 'CHM_ALS', 'CANOPY_DENSITY', 'slope', 'aspect', 'tpi',
       'tri', 'hillshade', 'LST'], dtype='<U14')

In [ ]:
MODEL_NAME = f"XGB_v1_{datetime.now().timestamp()}"
model = XGBRegressor(n_estimators=100, max_depth=16)
model.fit(train[SELECTED_PREDICTORS], train[LABEL])

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.core.SingleBatchInternalIter object at 0x00000158430E4290>>
Traceback (most recent call last):
  File "c:\Users\ramiq\Application\opengeohub-summerschool-2026\.venv\Lib\site-packages\xgboost\core.py", line 425, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument

KeyboardInterrupt: 


: 

In [ ]:
total = sum(model.feature_importances_)
print("Feature importance")
pd.Series(dict(zip(model.feature_names_in_, model.feature_importances_ / total * 100)))

Feature importance


BLUE               0.242525
GREEN              0.504253
RED                0.349933
NIR                0.377187
SWIR1              0.609990
SWIR2              0.626907
VV                 0.398074
VH                 0.416756
NDVI               0.606118
NDMI               0.778417
NBR                1.938790
NBR2               0.779764
NDWI              13.514425
MNDWI              0.480236
MNDWI2             0.474564
RVI                0.315933
CHM_META          63.295818
CHM_ALS            9.165105
CANOPY_DENSITY     0.997004
slope              0.491941
aspect             0.587686
tpi                0.380980
tri                0.797671
hillshade          0.680265
LST                1.189660
dtype: float32

In [ ]:
test_apply = model.predict(test[SELECTED_PREDICTORS])
r2 = np.corrcoef(test[LABEL], test_apply)[0, 1] ** 2
mae = mean_absolute_error(test[LABEL], test_apply)
rmse = root_mean_squared_error(test[LABEL], test_apply)
print(f"R^2={r2}", f"MAE={mae}", f"RMSE={rmse}")

R^2=0.8933317306653495 MAE=11.072398860238259 RMSE=16.933525847765026


In [ ]:
test_df = gpd.read_parquet(EXTRACT_TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,...,elevation,CANOPY_DENSITY,CHM_META,CHM_ALS,slope,aspect,tri,tpi,hillshade,LST
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,...,727,93,17,30.964409,7.375380,163.610458,6.782330,-1.250,170.0,23.340981
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,...,737,94,20,26.640724,6.659468,0.000000,5.196152,-0.375,186.0,27.931381
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,...,725,97,10,28.928268,4.745601,15.255112,4.795832,-1.125,182.0,23.135900
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,...,727,92,23,32.106686,3.169812,23.198593,3.162278,-0.500,180.0,23.108555
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,...,732,98,17,29.213942,5.584134,206.565048,4.472136,0.500,181.0,24.838074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),289.0,583.0,396.0,4034.0,...,1025,76,0,9.276945,2.943097,8.130104,2.236068,0.125,182.0,36.555046
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),686.0,1111.0,1201.0,4128.0,...,1023,53,0,16.030308,6.864422,345.963745,5.291502,-0.250,189.0,38.110245
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),286.0,574.0,375.0,3594.0,...,1025,77,11,8.037519,4.162167,0.000000,3.464102,0.000,184.0,38.110245
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),210.0,405.0,271.0,3095.0,...,1022,70,14,16.777363,4.120336,315.000000,3.605551,-0.625,187.0,38.086319


In [ ]:
generate_indices(test_df)
# generate_indices(test_df, "_DIST")
test_df

,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,...,hillshade,LST,NDVI,NDMI,NBR,NBR2,NDWI,MNDWI,MNDWI2,RVI
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,...,170.0,23.340981,9136.363636,4451.834370,7144.311530,3948.220065,-8274.552360,-6052.141527,-2764.505119,6313.488059
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,...,186.0,27.931381,8949.024544,4112.959925,6963.380282,3994.428969,-8089.516371,-5959.339263,-2578.763127,5957.873621
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,...,182.0,23.135900,9287.605295,5477.018154,7836.634765,4133.977067,-8478.210745,-5602.931379,-1911.764706,5052.573326
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,...,180.0,23.108555,9014.167651,4192.553426,7042.328042,4043.686734,-8146.478873,-6004.857316,-2590.090090,5794.743429
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,...,181.0,24.838074,9428.704134,5599.477904,8131.772865,4649.286158,-8762.054272,-6208.791209,-2192.362093,6947.230702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),289.0,583.0,396.0,4034.0,...,182.0,36.555046,8212.189616,3183.006536,5491.551459,2797.546012,-7474.550574,-5631.322593,-3363.688105,5631.067961
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),686.0,1111.0,1201.0,4128.0,...,189.0,38.110245,5492.587728,1806.091806,3446.254072,1749.026041,-5758.732583,-4411.468813,-2885.046430,5997.161107
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),286.0,574.0,375.0,3594.0,...,184.0,38.110245,8110.355253,2575.227432,4937.655860,2706.586826,-7245.681382,-5741.839763,-3593.750000,5781.990521
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),210.0,405.0,271.0,3095.0,...,187.0,38.086319,8389.780154,3823.135328,6454.013822,3492.682927,-7685.714286,-5469.798658,-2444.029851,5522.914219


In [ ]:
submission_df = pd.read_csv(SUBMISSION_CSV)
submission_df

,Id,Expected
0,028631_3040064_2251003,0.0
1,028631_3041214_2252063,0.0
2,028631_3041184_2251693,0.0
3,028631_3039864_2250783,0.0
4,028631_3040224_2251483,0.0
...,...,...
45014,061751_3374644_2043173,0.0
45015,061751_3374594_2043203,0.0
45016,061751_3374624_2043173,0.0
45017,061751_3374614_2043223,0.0


In [ ]:
submission_df["Expected"] = model.predict(test_df[SELECTED_PREDICTORS])
submission_df


,Id,Expected
0,028631_3040064_2251003,344.673615
1,028631_3041214_2252063,334.584869
2,028631_3041184_2251693,288.061218
3,028631_3039864_2250783,355.085632
4,028631_3040224_2251483,361.228790
...,...,...
45014,061751_3374644_2043173,50.205017
45015,061751_3374594_2043203,136.901108
45016,061751_3374624_2043173,110.572304
45017,061751_3374614_2043223,192.013794


In [ ]:
RESULT_CSV = f"{OUTPUT_PREFIX}/results_{MODEL_NAME}.csv"
submission_df.to_csv(RESULT_CSV, index=False)

MODEL_NAME

'XGB_v1_1787234394.925927'